In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from models.wave2vec2.audio import to_tensors

/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-12-05 04:06:58.774714: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-05 04:06:58.777577: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-12-05 04:06:58.777587: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [35]:
import os
import pickle

class AudioSegmentDataset(Dataset):
    def __init__(self, label_dir, input_dir, cache_dir, num_samples=None):
        self.label_dir = label_dir
        self.input_dir = input_dir
        self.cache_dir = cache_dir

        # Get all CSV files
        self.csv_files = os.listdir(label_dir)
        if num_samples:
            self.csv_files = self.csv_files[:num_samples]

        # Create the cache directory if it doesn't exist
        if not os.path.exists(cache_dir):
            os.makedirs(cache_dir)

    def _load_audio_features(self, file_name):
        cache_file = os.path.join(self.cache_dir, f"{file_name}.pkl")
        if os.path.exists(cache_file):
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        else:
            audio_file = os.path.join(self.input_dir, f"{file_name}.mp3")
            feature_vectors = to_tensors(audio_file, segment_length_ms=50)
            with open(cache_file, 'wb') as f:
                pickle.dump(feature_vectors, f)
            return feature_vectors

    def __len__(self):
        return len(self.csv_files)

    def __getitem__(self, idx):
        file = self.csv_files[idx]
        file_path = os.path.join(self.label_dir, file)
        file_name = os.path.basename(file).split(".")[0]

        # Load labels
        df = pd.read_csv(file_path)

        # Load audio features from cache
        feature_vectors = self._load_audio_features(file_name)

        # Store features and labels
        return {
            'features': feature_vectors,
            'breaks': torch.tensor(df['break'].values, dtype=torch.float32),
            'start_times': df['start'].values,
            'end_times': df['end'].values
        }

def create_dataloaders(label_dir, input_dir, cache_dir, num_samples=None, batch_size=1):
    dataset = AudioSegmentDataset(label_dir, input_dir, cache_dir, num_samples)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # train/test/validate split (90/5/5) split
    train_size = int(0.9 * len(dataloader))
    test_size = int(0.05 * len(dataloader))
    val_size = len(dataloader) - train_size - test_size
    
    train_dataloader, val_dataloader, test_dataloader = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size]
    )
    
    print(f"Train size: {len(train_dataloader)}")
    print(f"Val size: {len(val_dataloader)}")
    print(f"Test size: {len(test_dataloader)}")
    
    return train_dataloader, val_dataloader, test_dataloader


In [36]:
data_dir = '../../data'
label_dir = f'{data_dir}/clean/segment_breaks'
input_dir = f'{data_dir}/clean/audio/vocals'
cache_dir = f'cache'

train_dataloader, val_dataloader, test_dataloader = create_dataloaders(
    label_dir=label_dir,
    input_dir=input_dir,
    cache_dir=cache_dir,
    num_samples=None,  # if None, all samples will be used
    batch_size=1
)

Train size: 59
Val size: 4
Test size: 3


In [32]:
# print information about all the samples
for sample in train_dataloader:
    print("===============================")
    print(f"Sample features shape: {sample['features'].shape}")
    print(f"Sample breaks shape: {sample['breaks'].shape}")

Sample features shape: torch.Size([6322, 768])
Sample breaks shape: torch.Size([6323])
Sample features shape: torch.Size([5078, 768])
Sample breaks shape: torch.Size([5079])
Sample features shape: torch.Size([4410, 768])
Sample breaks shape: torch.Size([4411])
Sample features shape: torch.Size([4113, 768])
Sample breaks shape: torch.Size([4115])
Sample features shape: torch.Size([4115, 768])
Sample breaks shape: torch.Size([4116])
Sample features shape: torch.Size([4362, 768])
Sample breaks shape: torch.Size([4363])
Sample features shape: torch.Size([5299, 768])
Sample breaks shape: torch.Size([5301])
Sample features shape: torch.Size([4999, 768])
Sample breaks shape: torch.Size([5001])
Sample features shape: torch.Size([5263, 768])
Sample breaks shape: torch.Size([5265])
Sample features shape: torch.Size([4415, 768])
Sample breaks shape: torch.Size([4416])
Sample features shape: torch.Size([5254, 768])
Sample breaks shape: torch.Size([5255])
Sample features shape: torch.Size([4720, 76

In [96]:
# print information about all the samples
for sample in test_dataloader:
    print("===============================")
    print(f"Sample features shape: {sample['features'].shape}")
    print(f"Sample breaks shape: {sample['breaks'].shape}")

Sample features shape: torch.Size([6322, 768])
Sample breaks shape: torch.Size([6323])
Sample features shape: torch.Size([4410, 768])
Sample breaks shape: torch.Size([4411])
Sample features shape: torch.Size([5263, 768])
Sample breaks shape: torch.Size([5265])


In [60]:
class RNNModel(nn.Module):
    def __init__(self, input_size=768, hidden_size=256, num_layers=1, dropout=0, bidirectional=False):
        super(RNNModel, self).__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout, bidirectional=bidirectional)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), 1)

    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), self.rnn.hidden_size).to(x.device)
        c0 = torch.zeros(1, x.size(0), self.rnn.hidden_size).to(x.device)
        out, _ = self.rnn(x.unsqueeze(1), (h0, c0))
        out = self.fc(out)
        out = torch.sigmoid(out)
        return out

In [153]:
import torch
import torch.nn.functional as F

class CustomLoss(torch.nn.Module):
    def __init__(self, alpha=0.5, beta=1.0):
        """
        Custom loss function combining BCE and proximity loss
        
        Args:
        - alpha (float): Weight for BCE loss (0 <= alpha <= 1)
        - beta (float): Proximity loss sensitivity parameter
        """
        super(CustomLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        """
        Compute custom loss
        
        Args:
        - y_pred (torch.Tensor): Predicted break probabilities (batch x time_steps)
        - y_true (torch.Tensor): Ground truth break labels (batch x time_steps)
        
        Returns:
        - torch.Tensor: Computed loss
        """
        # BCE Loss
        f_bce = F.binary_cross_entropy(y_pred, y_true, reduction='sum')

        # Proximity Loss
        #f_proximity = -self._compute_proximity_loss(y_pred, y_true)  # TODO: uncomment this line

        # Total Loss
        f_total = self.alpha * f_bce #+ (1 - self.alpha) * f_proximity  # TODO: uncomment this line

        return f_total

    def _compute_proximity_loss(self, y_pred, y_true):
        """
        Compute proximity loss
        
        Args:
        - y_pred (torch.Tensor): Predicted break probabilities
        - y_true (torch.Tensor): Ground truth break labels
        
        Returns:
        - torch.Tensor: Proximity loss
        """
        # Find indices of predicted and true breaks
        pred_breaks = torch.where(y_pred > 0.5)[0]
        true_breaks = torch.where(y_true > 0.5)[0]

        # If no breaks predicted or no true breaks, return 0
        if len(pred_breaks) == 0 or len(true_breaks) == 0:
            return torch.tensor(0.0, device=y_pred.device)

        # Compute proximity loss
        proximity_loss = 0.0
        for t in pred_breaks:
            # Find closest true break
            distances = torch.abs(t - true_breaks)
            min_distance = torch.min(distances)

            # Compute proximity term
            proximity_term = torch.exp(self.beta * min_distance) - 1
            proximity_loss += proximity_term

        return proximity_loss


In [144]:
def adjust_size(y_pred, y_true):
    size_diff = y_true.size(0) - y_pred.size(0)
    if size_diff > 0:
        y_true = y_true.squeeze()[:-size_diff]
    else:
        y_true = y_true.squeeze()
    y_pred = y_pred.squeeze()
    return y_pred, y_true

In [145]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            features = batch['features']
            targets = batch['breaks']

            optimizer.zero_grad()
            outputs = model(features)

            # Compute loss using custom loss function
            loss = criterion(*adjust_size(outputs, targets))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_loss:.4f}")

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                features = batch['features']
                targets = batch['breaks']

                outputs = model(features)

                # Compute loss using custom loss function
                loss = criterion(*adjust_size(outputs, targets))
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Epoch {epoch+1}/{epochs}, Val Loss: {avg_val_loss:.4f}")


In [146]:
print(f"Initializing the model")
model = RNNModel()
# criterion = torch.nn.BCELoss()
criterion = CustomLoss(alpha=0.5, beta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Initializing the model


In [147]:
train_model(model, train_dataloader, val_dataloader, criterion, optimizer, epochs=10)

Epoch 1/10, Train Loss: 787.4462
Epoch 1/10, Val Loss: 703.5187
Epoch 2/10, Train Loss: 689.3406
Epoch 2/10, Val Loss: 691.8412
Epoch 3/10, Train Loss: 673.9465
Epoch 3/10, Val Loss: 692.4430
Epoch 4/10, Train Loss: 663.8246
Epoch 4/10, Val Loss: 694.6607
Epoch 5/10, Train Loss: 656.1314
Epoch 5/10, Val Loss: 696.5858
Epoch 6/10, Train Loss: 650.0206
Epoch 6/10, Val Loss: 698.2745
Epoch 7/10, Train Loss: 645.0010
Epoch 7/10, Val Loss: 700.1082
Epoch 8/10, Train Loss: 640.7353
Epoch 8/10, Val Loss: 702.0545
Epoch 9/10, Train Loss: 636.9730
Epoch 9/10, Val Loss: 703.8829
Epoch 10/10, Train Loss: 633.5498
Epoch 10/10, Val Loss: 705.3658


In [148]:
torch.save(model.state_dict(), 'model.pth')

In [151]:
def evaluate_model(model, test_loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            features = batch['features']
            targets = batch['breaks']

            outputs = model(features)

            # Compute loss using custom loss function
            loss = criterion(*adjust_size(outputs, targets))
            total_loss += loss.item()

    avg_loss = total_loss / len(test_loader)
    print(f"Test Loss: {avg_loss:.4f}")
    return avg_loss


In [152]:
evaluate_model(model, test_dataloader, criterion)

Test Loss: 719.7262


719.7261555989584